## Random Forest

In [0]:
train = spark.read.table("revenue_operations.gold.delivery_risk_train")
val = spark.read.table("revenue_operations.gold.delivery_risk_val")
test = spark.read.table("revenue_operations.gold.delivery_risk_test")

### ML Flow Experiment Setup and Scaling training set

In [0]:
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, precision_recall_curve, auc, confusion_matrix, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Mlflow setup dynamically
current_user = spark.sql("SELECT current_user()").collect()[0][0]
experiment_name = f"/Users/{current_user}/revenue_operations/delivery_risk"
mlflow.set_experiment(experiment_name)

print(f"Using experiment : {experiment_name}")

# Helper function to expand Spark ML vector columns into individual binary columns
from pyspark.ml.functions import vector_to_array

def expand_vectors(df):
    """Expand Spark ML vector columns into separate binary columns"""
    # Expand both vector columns at once (more efficient than multiple withColumn calls)
    df = df.withColumns({
        "customer_state_array": vector_to_array("customer_state_encoded"),
        "product_category_array": vector_to_array("primary_product_category_encoded")
    })
    
    # Drop original vector columns
    df = df.drop("customer_state_encoded", "primary_product_category_encoded")
    return df

print("Expanding encoded vector columns...")
train_expanded = expand_vectors(train)
val_expanded = expand_vectors(val)
print("✓ Vector columns expanded")

# Prepare data for sklearn by dropping non-numeric columns
# Exclude: order_id (ID), late_delivery_flag_indexed (target), timestamps
exclude_cols = [
    "order_id", 
    "late_delivery_flag_indexed",
    "order_approved_at", 
    "earliest_shipping_limit_date"
]
feature_cols = [col for col in train_expanded.columns if col not in exclude_cols]

print("\nConverting to pandas...")
# Converting Spark Dataframes to pandas
X_train = train_expanded.select(feature_cols).toPandas()
y_train = train_expanded.select("late_delivery_flag_indexed").toPandas().values.ravel()

X_val = val_expanded.select(feature_cols).toPandas()
y_val = val_expanded.select("late_delivery_flag_indexed").toPandas().values.ravel()

# Expand customer_state_array into separate columns
customer_state_df = pd.DataFrame(
    X_train['customer_state_array'].tolist(),
    columns=[f'customer_state_{i}' for i in range(len(X_train['customer_state_array'].iloc[0]))]
)

# Expand product_category_array into separate columns
product_category_df = pd.DataFrame(
    X_train['product_category_array'].tolist(),
    columns=[f'product_category_{i}' for i in range(len(X_train['product_category_array'].iloc[0]))]
)

# Drop array columns and concatenate expanded columns
X_train = X_train.drop(['customer_state_array', 'product_category_array'], axis=1)
X_train = pd.concat([X_train, customer_state_df, product_category_df], axis=1)

# Do the same for validation set
customer_state_df_val = pd.DataFrame(
    X_val['customer_state_array'].tolist(),
    columns=[f'customer_state_{i}' for i in range(len(X_val['customer_state_array'].iloc[0]))]
)

product_category_df_val = pd.DataFrame(
    X_val['product_category_array'].tolist(),
    columns=[f'product_category_{i}' for i in range(len(X_val['product_category_array'].iloc[0]))]
)

X_val = X_val.drop(['customer_state_array', 'product_category_array'], axis=1)
X_val = pd.concat([X_val, customer_state_df_val, product_category_df_val], axis=1)

print(f"✓ Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"✓ Validation set: {X_val.shape[0]} samples, {X_val.shape[1]} features")
print(f"  (includes {customer_state_df.shape[1]} customer state features + {product_category_df.shape[1]} product category features)")

# Convert any Decimal columns to float (required for MLflow serialization)
from decimal import Decimal
for col in X_train.columns:
    if X_train[col].dtype == 'object' and isinstance(X_train[col].iloc[0], Decimal):
        X_train[col] = X_train[col].astype(float)
        X_val[col] = X_val[col].astype(float)
print("✓ Converted Decimal columns to float")

# Scaling the X_train and X_val
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

### Random Forest Setup

In [0]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
# Suppress MLflow py4j warnings on serverless compute
import warnings
import logging

warnings.filterwarnings('ignore', category=UserWarning, module='mlflow')
logging.getLogger('mlflow').setLevel(logging.ERROR)
logging.getLogger('py4j').setLevel(logging.ERROR)

# Hyper paremters range
param_grid = {
    "n_estimators" : [250, 500],
    "max_depth" : [10, None],
    "min_samples_split" : [2, 5],
    "min_samples_leaf" : [2],
    "max_features" : ["sqrt", None]
}

# Create the model with class_weight='balanced' to handle imbalance
base_rf = RandomForestClassifier(random_state=42, class_weight='balanced')

# Set up grid search with 5 fold CV
random_search = RandomizedSearchCV(
    estimator = base_rf,
    param_distributions = param_grid,
    n_iter = 20,
    random_state = 42,
    cv = 3,
    scoring = "roc_auc",
    n_jobs = -1,
    verbose = 2
)

with mlflow.start_run(run_name = "random_forest_with_tuning"):

    # Fit the grid search
    random_search.fit(X_train_scaled, y_train)
    
    # Get the best results
    best_model = random_search.best_estimator_
    best_params = random_search.best_params_

    # Log parameters to mlflow
    mlflow.log_params(best_params)
    mlflow.log_param("model_type", "random_forest")

    # Make prediction on validation set
    y_train_pred = best_model.predict(X_train_scaled)
    y_pred = best_model.predict(X_val_scaled)
    y_pred_proba = best_model.predict_proba(X_val_scaled)[:, 1]

     # Calculate training metrics
    train_accuracy = accuracy_score(y_train, y_train_pred)
    train_precision = precision_score(y_train, y_train_pred, pos_label = 1.0)
    train_recall = recall_score(y_train, y_train_pred, pos_label = 1.0)
    train_f1 = f1_score(y_train, y_train_pred, pos_label = 1.0)

    # Calculate validation metrics
    accuracy = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred, pos_label = 1.0)
    recall = recall_score(y_val, y_pred, pos_label = 1.0)
    f1 = f1_score(y_val, y_pred, pos_label = 1.0)
    roc_auc = roc_auc_score(y_val, y_pred_proba)

    # Calculate PR curve
    precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_val, y_pred_proba, pos_label = 1.0)
    pr_auc = auc(recall_curve, precision_curve)

    # Business Metric Top 10% Late Order Capture
    # Sort by predicted probability descending
    sorted_indeces = np.argsort(-y_pred_proba)
    top_10_pct_count = int(len(y_val) * 0.1)
    top_10_pct_indeces = sorted_indeces[:top_10_pct_count]
    
    # How many actual late orders are in the top 10%?
    late_orders_in_top_10 = (y_val[top_10_pct_indeces] == 1.0).sum()
    total_late_orders = (y_val == 1.0).sum()
    capture_rate_10 = late_orders_in_top_10 / total_late_orders

    # Log to Mlflow with "train_" prefix
    mlflow.log_metric("train_accuracy", train_accuracy)
    mlflow.log_metric("train_precision_late", train_precision)
    mlflow.log_metric("train_recall_late", train_recall)
    mlflow.log_metric("train_f1_late", train_f1)

    # Log to Mlflow with "test_" prefix
    mlflow.log_metric("val_accuracy", accuracy)
    mlflow.log_metric("val_precision_late", precision)
    mlflow.log_metric("val_recall_late", recall)
    mlflow.log_metric("val_f1_late", f1)
    mlflow.log_metric("val_roc_auc", roc_auc)
    mlflow.log_metric("val_pr_auc", pr_auc)
    mlflow.log_metric("val_top10pct_capture", capture_rate_10)

    # Log model
    mlflow.sklearn.log_model(
        best_model, "model",
        input_example = X_train[:5]  # Use original pandas DataFrame, not scaled numpy array
    )

    # Log confusion matrix as artifact
    fig, ax = plt.subplots(figsize = (8,6))
    ConfusionMatrixDisplay.from_predictions(
        y_val, y_pred,
        display_labels = ["On-time", "Late"],
        cmap = "Blues",
        ax = ax
    )
    plt.title("Random Forest Confusion Matrix")
    mlflow.log_figure(fig, "confison_matrix_random_for.png")
    plt.close()

    # Generate ROC curve plot
    fpr, tpr, threshold = roc_curve(y_val, y_pred_proba, pos_label = 1.0)
    fig, ax = plt.subplots(figsize = (8,6))
    ax.plot(fpr, tpr, label = f'ROC AUC = {roc_auc:.4f}')
    ax.plot([0,1],[0,1], 'k--', label = "Random")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("Random Forest ROC Curve")
    ax.legend(loc = "lower right")
    mlflow.log_figure(fig, "roc_curve_random_for.png")
    plt.show()  
    plt.close()

    # Generate 
    fig, ax = plt.subplots(figsize = (8,6))
    ax.plot(recall_curve, precision_curve, label = f'PR AUC = {pr_auc:.4f}')
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Random Forest Precision-Recall Curve")
    ax.legend(loc = "lower left")
    mlflow.log_figure(fig, "pr_curve_random_for.png")
    plt.show()  
    plt.close()

    # Get run info
    active_run = mlflow.active_run()
    run_id = active_run.info.run_id if active_run else "unknown"

    # Display results
    print("\n" + "="*40)
    print("Random Forest Results")
    print("="*40)
    print(f"Run ID: {run_id}")
    print(f"\nMetrics:")
    print(f"  Accuracy:           {accuracy:.4f}")
    print(f"  Precision (Late):   {precision:.4f}")
    print(f"  Recall (Late):      {recall:.4f}")
    print(f"  F1 Score (Late):    {f1:.4f}")
    print(f"  ROC-AUC:            {roc_auc:.4f}")
    print(f"  PR-AUC:             {pr_auc:.4f}")
    print(f"  Top 10% Capture:    {capture_rate_10:.1%}")
    print(f"    → Found {late_orders_in_top_10} late orders in top {top_10_pct_count:,} highest-risk orders")
    print(f"    → Out of {total_late_orders} total late orders in validation set")
    print("\nClassification Report:")
    print(classification_report(y_val, y_pred, target_names=["On-time", "Late"]))
    print("="*50)

    # Display confusion matrix
    ConfusionMatrixDisplay.from_predictions(
        y_val, y_pred,
        display_labels = ["On-time", "Late"],
        cmap = "Blues"
    )
    plt.title("Random Forest Confusion Matrix")
    plt.show()

print(f"\n Results saved to Mlflow experiment: {experiment_name}")
print(f"View in Mlflow UI")